In [1]:
# importing the essential libraries
import numpy as np
import pandas as pd

In [15]:
# loading the datasets
transactions_data = pd.read_csv('../../data/transactions_clean.csv')
inventory_data = pd.read_csv('../../data/inventory_clean.csv')
suppliers_data = pd.read_csv('../../data/suppliers.csv')
products_data = pd.read_csv('../../data/products.csv')

In [16]:
# merging the datasets
merged_df = pd.merge(transactions_data, inventory_data, on=['product_id', 'date'])
product_cols = ['product_id', 'unit_price', 'unit_cost', 'category', 'base_demand']
merged_df_a = pd.merge(merged_df, products_data[product_cols], on=['product_id'])
master_df = pd.merge(merged_df_a, suppliers_data, on=['supplier_id'])

In [17]:
print(f"Master DataFrame Shape: {master_df.shape}")

Master DataFrame Shape: (1825, 16)


## Master DataFrame Merging Summary

### 1. Merge Strategy & Integrity Check
* **Composite Core Join:** Merged `transactions_clean.csv` and `inventory_clean.csv` using a composite inner join on both `date` and `product_id` to cleanly line up daily data without duplicate generation.
* **Targeted Metadata Mappings:** Merged `products.csv` while explicitly filtering for only the requested columns (`unit_price`, `unit_cost`, `category`, `base_demand`) along with the `supplier_id` key from the suppliers reference table.
* **Shape Validation:** The resulting master DataFrame contains exactly **1,825 rows**, confirming perfect structural alignment with no duplicate rows.

### 2. Final Column Schema Map
The master dataset is comprised of the following **12 columns** used across this entire bivariate analysis section:

* `date` (Datetime key)
* `product_id` (Product key)
* `units_sold` (Daily sales volume)
* `revenue` (Gross dollar sales)
* `cogs` (Cost of goods sold)
* `gross_profit` (Net margin dollar profit)
* `closing_stock` (Warehouse stock remaining)
* `stockout_flag` (Binary stockout event indicator)
* `unit_price` (Retail item price)
* `unit_cost` (Wholesale item cost)
* `category` (Item department grouping)
* `base_demand` (Configured baseline demand setting)
* `supplier_id` (Supplier key)
* `supplier_name` (Supplier organization name)
* `lead_time_days` (Fulfillment delivery time)
* `reliability` (Supplier fulfillment accuracy rate)

In [18]:
cols = ['units_sold', 'revenue', 'gross_profit', 'closing_stock', 'stockout_flag', 'lead_time_days', 'reliability', 'base_demand', 'unit_price']
correlation_matrix = master_df[cols].corr().round(3)
correlation_matrix

,units_sold,revenue,gross_profit,closing_stock,stockout_flag,lead_time_days,reliability,base_demand,unit_price
units_sold,1.000,0.249,0.254,-0.165,0.080,-0.235,0.182,0.736,-0.505
revenue,0.249,1.000,0.992,-0.246,0.314,0.507,-0.505,-0.135,0.576
gross_profit,0.254,0.992,1.000,-0.215,0.272,0.418,-0.412,-0.161,0.601
closing_stock,-0.165,-0.246,-0.215,1.000,-0.535,-0.304,0.312,-0.056,0.026
stockout_flag,0.080,0.314,0.272,-0.535,1.000,0.389,-0.398,0.015,0.093
lead_time_days,-0.235,0.507,0.418,-0.304,0.389,1.000,-0.996,-0.348,0.355
reliability,0.182,-0.505,-0.412,0.312,-0.398,-0.996,1.000,0.279,-0.315
base_demand,0.736,-0.135,-0.161,-0.056,0.015,-0.348,0.279,1.000,-0.698
unit_price,-0.505,0.576,0.601,0.026,0.093,0.355,-0.315,-0.698,1.000


## Correlation Matrix

### 1. The Three Strongest Positive Relationships (Moving Together)

* **`revenue` vs. `gross_profit` (0.992) $\rightarrow$ Strong Positive**
  * **What it means:** These two are like twins. Whenever your sales revenue goes up, your cash profit goes up at almost the exact same rate. This means the business is making healthy profit margins on its sales.
* **`units_sold` vs. `base_demand` (0.736) $\rightarrow$ Strong Positive**
  * **What it means:** Products that were expected to sell in high volumes (`base_demand`) actually do sell in high volumes (`units_sold`). The initial setup matches real life.
* **`gross_profit` vs. `unit_price` (0.601) $\rightarrow$ Moderate Positive**
  * **What it means:** Products with higher price tags tend to bring in more daily profit dollars overall, even if they aren't sold quite as often as cheaper items.

---

### 2. The Three Strongest Negative Relationships (The See-Saws)

* **`lead_time_days` vs. `reliability` (-0.996) $\rightarrow$ Strong Negative**
  * **What it means:** A massive warning sign. As soon as a supplier's delivery time takes longer, their trustworthiness crashes. The slower, faraway suppliers are incredibly flaky.
* **`base_demand` vs. `unit_price` (-0.698) $\rightarrow$ Moderate Negative**
  * **What it means:** This shows how the items are priced. Cheap items have a high baseline demand (lots of people want them), while expensive premium items have a low baseline demand.
* **`closing_stock` vs. `stockout_flag` (-0.535) $\rightarrow$ Moderate Negative**
  * **What it means:** This just proves your logic makes sense. As the physical stock left in the warehouse drops toward zero, out-of-stock alarms start turning on.